#### FIFTH ATTEMPT

## Temporal Attack Prediction

### OpTC + Windows-APT datasets

**Overview:**

The combined dataset integrates OpTC endpoint telemetry with Windows-APT 2025 telemetry to provide a broader set of benign and attack-related system activities for temporal cyberattack prediction. The two datasets were harmonised into a common schema containing timestamp, event_action, event_object, protocol, process and thread identifiers (pid, ppid, tid), hostname, label, and dataset_source.
For the LSTM and GRU experiment, timestamps are used to chronologically organise events and construct temporal sequences, allowing the models to learn patterns in preceding system activity and predict whether malicious activity will occur within a subsequent time window.

**Project goal:**

The aim is temporal prediction rather than event-level detection: use activity from the previous 10 minutes to predict whether malicious activity will occur during the next 5 minutes. The raw timestamp is retained for chronological ordering and window construction. hour and minute are also retained as model features, consistent with the previous experiment. Windows are created separately within each dataset source, hostname and date so sequences do not cross hosts, datasets or day boundaries.



In [36]:
# Imports
from pathlib import Path
import gc
import random
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.preprocessing import StandardScaler
from sklearn.utils.class_weight import compute_class_weight
from sklearn.model_selection import GroupShuffleSplit
from sklearn.linear_model import LogisticRegression
from sklearn import metrics
from sklearn.model_selection import train_test_split
from xgboost import XGBClassifier

import tensorflow as tf
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Input, LSTM, GRU, Dense, Dropout, BatchNormalization
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau

warnings.filterwarnings("ignore", category=FutureWarning)

seed = 7
random.seed(seed)
np.random.seed(seed)
tf.random.set_seed(seed)

In [37]:
# Mount Google Drive
from google.colab import drive
drive.mount("/content/drive")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


### Create Temporal Combination Dataset

In [38]:
# Paths
optc_path = Path("/content/drive/MyDrive/solutions/OpTC_sample2_cleaned")
apt_path = Path("/content/drive/MyDrive/solutions/Windows_APT/combined.csv")
mapping_path = Path("/content/drive/MyDrive/solutions/Windows_APT/log_to_scenario_mapping.csv")

temporal_combination_path = Path(
    "/content/drive/MyDrive/solutions/Temporal_Combination_OpTC_APT"
)
temporal_combination_path.mkdir(parents=True, exist_ok=True)

TRAIN_PATH = temporal_combination_path / "temporal_train.parquet"
VALIDATION_PATH = temporal_combination_path / "temporal_validation.parquet"
TEST_PATH = temporal_combination_path / "temporal_test.parquet"

assert optc_path.exists(), f"OpTC directory not found: {optc_path}"
assert apt_path.exists(), f"Windows-APT CSV not found: {apt_path}"
assert mapping_path.exists(), f"Mapping CSV not found: {mapping_path}"

In [39]:
# # Helper functions used during dataset creation
# def parse_timestamp_by_source(df):
#     result = pd.Series(pd.NaT, index=df.index, dtype="datetime64[ns, UTC]")

#     apt_mask = df["dataset_source"].eq("Windows-APT")
#     if apt_mask.any():
#         apt_raw = df.loc[apt_mask, "timestamp"].astype("string")
#         parsed = pd.to_datetime(
#             apt_raw,
#             format="%b %d, %Y @ %H:%M:%S.%f",
#             errors="coerce",
#             utc=True,
#         )
#         # Fallback for valid ISO or slightly different timestamp strings.
#         missing = parsed.isna()
#         if missing.any():
#             parsed.loc[missing] = pd.to_datetime(
#                 apt_raw.loc[missing], errors="coerce", utc=True
#             )
#         result.loc[apt_mask] = parsed

#     optc_mask = df["dataset_source"].eq("OpTC")
#     if optc_mask.any():
#         optc_raw = df.loc[optc_mask, "timestamp"]
#         numeric = pd.to_numeric(optc_raw, errors="coerce")
#         numeric_mask = numeric.notna()

#         if numeric_mask.any():
#             values = numeric[numeric_mask]
#             median_value = values.abs().median()
#             if median_value > 1e17:
#                 unit = "ns"
#             elif median_value > 1e14:
#                 unit = "us"
#             elif median_value > 1e11:
#                 unit = "ms"
#             else:
#                 unit = "s"
#             result.loc[values.index] = pd.to_datetime(
#                 values, unit=unit, errors="coerce", utc=True
#             )

#         string_idx = optc_raw.index[~numeric_mask]
#         if len(string_idx):
#             result.loc[string_idx] = pd.to_datetime(
#                 optc_raw.loc[string_idx], errors="coerce", utc=True
#             )

#     return result


# def normalise_binary_label(series):
#     """Convert defensible binary label representations to 0/1.

#     Numeric 0/1 and descriptive benign/malicious values are supported.
#     Unrecognised values remain missing and are reported before modelling.
#     """
#     result = pd.Series(np.nan, index=series.index, dtype="float64")

#     numeric = pd.to_numeric(series, errors="coerce")
#     numeric_binary = numeric.isin([0, 1])
#     result.loc[numeric_binary] = numeric.loc[numeric_binary]

#     text = (
#         series.astype("string")
#         .str.strip()
#         .str.lower()
#     )

#     benign_pattern = r"benign|normal|clean|negative|non[-_ ]?attack|false"
#     attack_pattern = r"malicious|attack|positive|anomal|compromis|true|apt"

#     benign_mask = result.isna() & text.str.contains(
#         benign_pattern, regex=True, na=False
#     )
#     result.loc[benign_mask] = 0

#     attack_mask = result.isna() & text.str.contains(
#         attack_pattern, regex=True, na=False
#     )
#     result.loc[attack_mask] = 1

#     return result


# def standardise_common_columns(df):
#     df = df.copy()
#     df["hostname"] = (
#         df["hostname"].astype("string").fillna("").str.strip().str.lower()
#     )
#     for col in ["pid", "ppid", "tid"]:
#         df[col] = pd.to_numeric(df[col], errors="coerce").fillna(-1).astype("float32")
#     for col in ["event_action", "event_object", "protocol"]:
#         df[col] = df[col].astype("string").fillna("UNKNOWN").str.strip()
#         df.loc[df[col].eq(""), col] = "UNKNOWN"
#     df["label"] = pd.to_numeric(df["label"], errors="coerce")
#     return df


In [40]:
# # Load and standardise OpTC
# files = sorted(optc_path.glob("*.parquet"))
# assert files, f"No parquet files found in {optc_path}"

# selected_features = [
#     "action", "object", "l4protocol", "pid", "ppid", "tid",
#     "timestamp", "label", "hostname",
# ]

# data_parts = []
# for file in files:
#     part = pd.read_parquet(file, columns=selected_features)
#     data_parts.append(part)

# optc = pd.concat(data_parts, ignore_index=True)
# del data_parts, part
# gc.collect()

# optc.rename(
#     columns={"action": "event_action", "object": "event_object", "l4protocol": "protocol"},
#     inplace=True,
# )
# optc["dataset_source"] = "OpTC"
# optc["scenario_id"] = pd.NA
# optc = standardise_common_columns(optc)

# invalid_optc_labels = ~optc["label"].isin([0, 1])
# assert not invalid_optc_labels.any(), (
#     f"OpTC contains {invalid_optc_labels.sum():,} labels outside 0/1. "
#     "Resolve these labels before continuing."
# )
# optc["label"] = optc["label"].astype("int8")

# print("OpTC loaded:", optc.shape)
# print(optc["label"].value_counts(dropna=False))


In [41]:
# # Load Windows-APT and its row-level mapping

# apt = pd.read_csv(
#     apt_path,
#     low_memory=False
# )

# mapping = pd.read_csv(
#     mapping_path,
#     low_memory=False
# )


# # Verify that the mapping corresponds row-for-row
# # with the Windows-APT dataset

# assert len(apt) == len(mapping), (
#     "Windows-APT and mapping row counts differ: "
#     f"{len(apt):,} versus {len(mapping):,}. "
#     "Do not concatenate them until their row "
#     "identifiers have been matched."
# )

# assert "Derived_Label" in mapping.columns, (
#     "Derived_Label is missing from the mapping file."
# )


# print(
#     "Windows-APT Derived_Label values:"
# )

# print(
#     mapping[
#         "Derived_Label"
#     ]
#     .value_counts(
#         dropna=False
#     )
#     .head(20)
# )


# # Combine the Windows-APT events with
# # their corresponding scenario information

# apt_mapped = pd.concat(
#     [
#         apt.reset_index(
#             drop=True
#         ),

#         mapping[
#             [
#                 "Scenario_ID",
#                 "Scenario_Name",
#                 "Derived_Label"
#             ]
#         ].reset_index(
#             drop=True
#         )
#     ],
#     axis=1
# )


# del apt
# del mapping
# gc.collect()


# # Rename Windows-APT columns

# apt_mapped.rename(
#     columns={
#         "_source.@timestamp":
#             "timestamp",

#         "_source.data.win.system.computer":
#             "hostname",

#         "_source.data.win.eventdata.eventType":
#             "event_type",

#         "_source.data.win.eventdata.type":
#             "type",

#         "_source.data.win.eventdata.processId":
#             "pid",

#         "_source.data.win.eventdata.parentProcessId":
#             "ppid",

#         "_source.data.win.system.threadID":
#             "tid",

#         "_source.data.win.eventdata.protocol":
#             "protocol",

#         "Scenario_ID":
#             "scenario_id",

#         "Scenario_Name":
#             "scenario_name"
#     },
#     inplace=True
# )


# # Avoid using rule descriptions because they
# # are closely related to the security-rule label

# apt_mapped["event_action"] = (
#     apt_mapped.get(
#         "event_type",
#         pd.Series(
#             index=apt_mapped.index,
#             dtype="object"
#         )
#     )
# )

# apt_mapped["event_object"] = (
#     apt_mapped.get(
#         "type",
#         pd.Series(
#             index=apt_mapped.index,
#             dtype="object"
#         )
#     )
# )


# # Convert Derived_Label into a binary proxy label
# #
# # untagged = no technique tag
# # technique-tagged = technique successfully identified
# # technique-unmatched = technique evidence exists,
# #                       but was not matched to a scenario

# derived_label_mapping = {
#     "untagged": 0,
#     "technique-tagged": 1,
#     "technique-unmatched": 1
# }


# derived_label_clean = (
#     apt_mapped[
#         "Derived_Label"
#     ]
#     .astype("string")
#     .str.strip()
#     .str.lower()
# )


# apt_mapped["label"] = (
#     derived_label_clean
#     .map(
#         derived_label_mapping
#     )
# )


# print(
#     "\nWindows-APT label conversion:"
# )

# print(
#     pd.crosstab(
#         apt_mapped[
#             "Derived_Label"
#         ],

#         apt_mapped[
#             "label"
#         ],

#         dropna=False
#     )
# )


# # Identify missing or unexpected label values

# invalid_apt_labels = (
#     apt_mapped[
#         "label"
#     ].isna()
# )

# unresolved_count = int(
#     invalid_apt_labels.sum()
# )


# print(
#     "\nRows with missing or unresolved "
#     "Derived_Label:",
#     unresolved_count
# )


# if unresolved_count > 0:

#     print(
#         "\nUnresolved Derived_Label values:"
#     )

#     print(
#         apt_mapped.loc[
#             invalid_apt_labels,
#             "Derived_Label"
#         ]
#         .value_counts(
#             dropna=False
#         )
#     )

#     apt_mapped = (
#         apt_mapped.loc[
#             ~invalid_apt_labels
#         ]
#         .copy()
#     )


# # Add dataset source

# apt_mapped[
#     "dataset_source"
# ] = "Windows-APT"


# # Select the harmonised columns

# required_apt_columns = [
#     "timestamp",
#     "hostname",
#     "event_action",
#     "event_object",
#     "pid",
#     "ppid",
#     "tid",
#     "protocol",
#     "label",
#     "dataset_source",
#     "scenario_id"
# ]


# missing_apt_columns = [
#     column
#     for column in required_apt_columns
#     if column not in apt_mapped.columns
# ]


# assert not missing_apt_columns, (
#     "Missing Windows-APT columns: "
#     f"{missing_apt_columns}"
# )


# apt_mapped = standardise_common_columns(
#     apt_mapped[
#         required_apt_columns
#     ]
# )


# # Final label validation

# invalid_apt_labels = (
#     ~apt_mapped[
#         "label"
#     ].isin(
#         [
#             0,
#             1
#         ]
#     )
# )


# assert not invalid_apt_labels.any(), (
#     f"{invalid_apt_labels.sum():,} Windows-APT "
#     "rows still have invalid labels."
# )


# apt_mapped["label"] = (
#     apt_mapped[
#         "label"
#     ]
#     .astype("int8")
# )


# assert (
#     apt_mapped[
#         "label"
#     ].nunique()
#     == 2
# ), (
#     "Windows-APT must contain both "
#     "class 0 and class 1."
# )


# print(
#     "\nWindows-APT loaded:",
#     apt_mapped.shape
# )

# print(
#     "\nFinal Windows-APT label distribution:"
# )

# print(
#     apt_mapped[
#         "label"
#     ]
#     .value_counts(
#         dropna=False
#     )
# )

In [42]:
# # Combine, parse timestamps, remove invalid records and exact duplicates
# harmonised_features = [
#     "timestamp", "hostname", "event_action", "event_object", "pid", "ppid",
#     "tid", "protocol", "label", "dataset_source", "scenario_id",
# ]

# data = pd.concat(
#     [optc[harmonised_features], apt_mapped[harmonised_features]],
#     ignore_index=True,
# )
# del optc, apt_mapped
# gc.collect()

# data["timestamp"] = parse_timestamp_by_source(data)
# invalid_timestamps = data["timestamp"].isna()
# print(f"Invalid timestamps removed: {invalid_timestamps.sum():,}")
# data = data.loc[~invalid_timestamps].copy()

# rows_before = len(data)
# duplicate_mask = data.duplicated(subset=harmonised_features, keep="first")
# print(f"Exact duplicate events removed: {duplicate_mask.sum():,}")
# data = data.loc[~duplicate_mask].reset_index(drop=True)
# print(f"Rows before deduplication: {rows_before:,}")
# print(f"Rows after deduplication:  {len(data):,}")

# data["date"] = data["timestamp"].dt.strftime("%Y-%m-%d")

# # Independent splitting units: host for OpTC; scenario for Windows-APT.
# optc_mask = data["dataset_source"].eq("OpTC")
# data.loc[optc_mask, "split_group"] = (
#     "OpTC|host|" + data.loc[optc_mask, "hostname"].astype(str)
# )

# apt_mask = data["dataset_source"].eq("Windows-APT")
# apt_scenario = data.loc[apt_mask, "scenario_id"].astype("string").str.strip()
# valid_scenario = apt_scenario.notna() & ~apt_scenario.isin(["", "nan", "None", "UNRESOLVED"])

# apt_groups = pd.Series(index=apt_scenario.index, dtype="string")
# apt_groups.loc[valid_scenario] = "Windows-APT|scenario|" + apt_scenario.loc[valid_scenario]
# apt_groups.loc[~valid_scenario] = (
#     "Windows-APT|fallback|"
#     + data.loc[apt_groups.index[~valid_scenario], "hostname"].astype(str)
#     + "|"
#     + data.loc[apt_groups.index[~valid_scenario], "date"].astype(str)
# )
# data.loc[apt_mask, "split_group"] = apt_groups

# assert data["split_group"].notna().all()
# print("Combined cleaned dataset:", data.shape)
# print(pd.crosstab(data["dataset_source"], data["label"]))
# print("Independent groups by source:")
# print(data.groupby("dataset_source")["split_group"].nunique())


In [43]:
# # Fixed OpTC host split
# # Each host appears in exactly one partition.

# optc_train_hosts = [
#     "sysclient0501.systemia.com",
#     "sysclient0201.systemia.com",
#     "sysclient0075.systemia.com"
# ]

# optc_validation_hosts = [
#     "sysclient0351.systemia.com"
# ]

# optc_test_hosts = [
#     "sysclient0051.systemia.com",
#     "sysclient0352.systemia.com"
# ]


# # Separate the two data sources

# optc_data = data[
#     data["dataset_source"].eq(
#         "OpTC"
#     )
# ].copy()

# apt_data = data[
#     data["dataset_source"].eq(
#         "Windows-APT"
#     )
# ].copy()


# # Verify all specified OpTC hosts exist

# available_optc_hosts = set(
#     optc_data[
#         "hostname"
#     ].unique()
# )

# requested_optc_hosts = set(
#     optc_train_hosts
#     + optc_validation_hosts
#     + optc_test_hosts
# )

# missing_optc_hosts = (
#     requested_optc_hosts
#     - available_optc_hosts
# )

# assert not missing_optc_hosts, (
#     "The following specified OpTC hosts "
#     "were not found: "
#     + ", ".join(
#         sorted(
#             missing_optc_hosts
#         )
#     )
# )


# # Verify that the host lists do not overlap

# assert set(
#     optc_train_hosts
# ).isdisjoint(
#     set(
#         optc_validation_hosts
#     )
# )

# assert set(
#     optc_train_hosts
# ).isdisjoint(
#     set(
#         optc_test_hosts
#     )
# )

# assert set(
#     optc_validation_hosts
# ).isdisjoint(
#     set(
#         optc_test_hosts
#     )
# )


# # Create OpTC partitions

# optc_train = optc_data[
#     optc_data[
#         "hostname"
#     ].isin(
#         optc_train_hosts
#     )
# ].copy()

# optc_validation = optc_data[
#     optc_data[
#         "hostname"
#     ].isin(
#         optc_validation_hosts
#     )
# ].copy()

# optc_test = optc_data[
#     optc_data[
#         "hostname"
#     ].isin(
#         optc_test_hosts
#     )
# ].copy()


# # Fast group-based Windows-APT split

# def split_apt_groups(
#     source_data,
#     random_state=7
# ):

#     group_summary = (
#         source_data
#         .groupby(
#             "split_group",
#             observed=True
#         )
#         .agg(
#             group_label=(
#                 "label",
#                 "max"
#             )
#         )
#         .reset_index()
#     )

#     assert len(group_summary) >= 3, (
#         "Windows-APT requires at least "
#         "three independent scenario groups."
#     )


#     # Development/test split

#     stratify_labels = None

#     group_label_counts = (
#         group_summary[
#             "group_label"
#         ]
#         .value_counts()
#     )

#     if (
#         len(group_label_counts) > 1
#         and group_label_counts.min() >= 2
#     ):

#         stratify_labels = (
#             group_summary[
#                 "group_label"
#             ]
#         )


#     development_groups, test_groups = (
#         train_test_split(
#             group_summary,
#             test_size=0.20,
#             random_state=random_state,
#             stratify=stratify_labels
#         )
#     )


#     # Train/validation split

#     development_stratify = None

#     development_label_counts = (
#         development_groups[
#             "group_label"
#         ]
#         .value_counts()
#     )

#     if (
#         len(development_label_counts) > 1
#         and development_label_counts.min() >= 2
#     ):

#         development_stratify = (
#             development_groups[
#                 "group_label"
#             ]
#         )


#     train_groups, validation_groups = (
#         train_test_split(
#             development_groups,
#             test_size=0.25,
#             random_state=random_state + 1,
#             stratify=development_stratify
#         )
#     )


#     train_group_names = set(
#         train_groups[
#             "split_group"
#         ]
#     )

#     validation_group_names = set(
#         validation_groups[
#             "split_group"
#         ]
#     )

#     test_group_names = set(
#         test_groups[
#             "split_group"
#         ]
#     )


#     train_part = source_data[
#         source_data[
#             "split_group"
#         ].isin(
#             train_group_names
#         )
#     ].copy()


#     validation_part = source_data[
#         source_data[
#             "split_group"
#         ].isin(
#             validation_group_names
#         )
#     ].copy()


#     test_part = source_data[
#         source_data[
#             "split_group"
#         ].isin(
#             test_group_names
#         )
#     ].copy()


#     return (
#         train_part,
#         validation_part,
#         test_part
#     )


# # Split Windows-APT by scenario groups

# apt_train, apt_validation, apt_test = (
#     split_apt_groups(
#         apt_data,
#         random_state=seed
#     )
# )


# # Combine the corresponding source partitions

# train_data = pd.concat(
#     [
#         optc_train,
#         apt_train
#     ],
#     ignore_index=True
# )

# validation_data = pd.concat(
#     [
#         optc_validation,
#         apt_validation
#     ],
#     ignore_index=True
# )

# test_data = pd.concat(
#     [
#         optc_test,
#         apt_test
#     ],
#     ignore_index=True
# )


# # Verify group independence

# train_groups = set(
#     train_data[
#         "split_group"
#     ].unique()
# )

# validation_groups = set(
#     validation_data[
#         "split_group"
#     ].unique()
# )

# test_groups = set(
#     test_data[
#         "split_group"
#     ].unique()
# )


# assert train_groups.isdisjoint(
#     validation_groups
# ), (
#     "Train and validation groups overlap."
# )

# assert train_groups.isdisjoint(
#     test_groups
# ), (
#     "Train and test groups overlap."
# )

# assert validation_groups.isdisjoint(
#     test_groups
# ), (
#     "Validation and test groups overlap."
# )


# # Show partition information

# for name, split_data in {
#     "TRAIN": train_data,
#     "VALIDATION": validation_data,
#     "TEST": test_data
# }.items():

#     print(
#         f"\n===== {name} ====="
#     )

#     print(
#         "Shape:",
#         split_data.shape
#     )

#     print(
#         "\nSource distribution:"
#     )

#     print(
#         split_data[
#             "dataset_source"
#         ].value_counts()
#     )

#     print(
#         "\nLabel distribution:"
#     )

#     print(
#         split_data[
#             "label"
#         ].value_counts()
#     )

#     print(
#         "\nSource and label:"
#     )

#     print(
#         pd.crosstab(
#             split_data[
#                 "dataset_source"
#             ],

#             split_data[
#                 "label"
#             ]
#         )
#     )

#     print(
#         "\nIndependent groups:",
#         split_data[
#             "split_group"
#         ].nunique()
#     )

In [44]:
# # Verify group independence and save datasets using the experiment folder/name
# train_groups = set(train_data["split_group"].unique())
# validation_groups = set(validation_data["split_group"].unique())
# test_groups = set(test_data["split_group"].unique())

# assert train_groups.isdisjoint(validation_groups)
# assert train_groups.isdisjoint(test_groups)
# assert validation_groups.isdisjoint(test_groups)

# for name, split_data in {
#     "TRAIN": train_data,
#     "VALIDATION": validation_data,
#     "TEST": test_data,
# }.items():
#     print(f"\n{name}: {split_data.shape}")
#     print(pd.crosstab(split_data["dataset_source"], split_data["label"], margins=True))
#     print("Positive rate:", split_data["label"].mean())
#     print("Groups:", split_data["split_group"].nunique())

# save_columns = harmonised_features + ["split_group"]
# train_data[save_columns].to_parquet(TRAIN_PATH, index=False)
# validation_data[save_columns].to_parquet(VALIDATION_PATH, index=False)
# test_data[save_columns].to_parquet(TEST_PATH, index=False)

# print("\nSaved:")
# print(TRAIN_PATH)
# print(VALIDATION_PATH)
# print(TEST_PATH)


# Deleting objects from file creation
# del data, train_data, validation_data, test_data
# gc.collect()



### Load Saved Datasets

In [45]:

train_data = pd.read_parquet(TRAIN_PATH)
validation_data = pd.read_parquet(VALIDATION_PATH)
test_data = pd.read_parquet(TEST_PATH)

for split_data in [train_data, validation_data, test_data]:
    split_data["timestamp"] = pd.to_datetime(split_data["timestamp"], errors="coerce", utc=True)
    assert split_data["timestamp"].notna().all()

print("Train shape:", train_data.shape)
print("Validation shape:", validation_data.shape)
print("Test shape:", test_data.shape)


Train shape: (4944347, 12)
Validation shape: (1416232, 12)
Test shape: (2811309, 12)


### Minute-level Feature Engineering

In [46]:
# Temporal settings
TIME_BIN = "1min"
LOOKBACK_MINUTES = 10
PREDICT_AHEAD_MINUTES = 3

TOP_ACTIONS = 30
TOP_OBJECTS = 20
TOP_PROTOCOLS = 10

top_actions = train_data["event_action"].value_counts().head(TOP_ACTIONS).index.tolist()
top_objects = train_data["event_object"].value_counts().head(TOP_OBJECTS).index.tolist()
top_protocols = train_data["protocol"].value_counts().head(TOP_PROTOCOLS).index.tolist()


def clean_feature_name(value):
    cleaned = "".join(ch if ch.isalnum() else "_" for ch in str(value).strip())
    return "_".join(part for part in cleaned.split("_") if part) or "UNKNOWN"


def unique_feature_mapping(values, prefix):
    result, used = {}, set()
    for position, value in enumerate(values):
        base = f"{prefix}__{clean_feature_name(value)}"
        name = base
        counter = 2
        while name in used:
            name = f"{base}_{counter}"
            counter += 1
        result[value] = name
        used.add(name)
    return result


action_columns = unique_feature_mapping(top_actions, "action")
object_columns = unique_feature_mapping(top_objects, "object")
protocol_columns = unique_feature_mapping(top_protocols, "protocol")


In [47]:
def build_minute_features(data):
    required = [
        "dataset_source", "hostname", "split_group", "timestamp", "pid", "ppid",
        "tid", "label", "event_action", "event_object", "protocol",
    ]
    missing = [c for c in required if c not in data.columns]
    assert not missing, f"Missing columns: {missing}"

    df = data[required].copy()
    df["time_bin"] = df["timestamp"].dt.floor(TIME_BIN)
    df["date"] = df["time_bin"].dt.strftime("%Y-%m-%d")
    keys = ["dataset_source", "hostname", "split_group", "date", "time_bin"]

    minute_data = (
        df.groupby(keys, sort=False, observed=True)
        .agg(
            event_count=("label", "size"),
            unique_pid=("pid", "nunique"),
            unique_ppid=("ppid", "nunique"),
            unique_tid=("tid", "nunique"),
            attack_now=("label", "max"),
        )
        .reset_index()
    )

    def add_counts(frame, category, vocabulary, rename_map):
        counts = frame.loc[frame[category].isin(vocabulary), keys + [category]]
        counts = counts.groupby(keys + [category], sort=False, observed=True).size()
        counts = counts.unstack(category, fill_value=0).rename(columns=rename_map).reset_index()
        return counts

    for category, vocabulary, rename_map in [
        ("event_action", top_actions, action_columns),
        ("event_object", top_objects, object_columns),
        ("protocol", top_protocols, protocol_columns),
    ]:
        counts = add_counts(df, category, vocabulary, rename_map)
        minute_data = minute_data.merge(counts, on=keys, how="left", validate="one_to_one")

    # Complete minute grid separately inside every source/host/group/date.
    completed = []
    group_keys = ["dataset_source", "hostname", "split_group", "date"]
    for group_values, group in minute_data.groupby(group_keys, sort=False, observed=True):
        group = group.sort_values("time_bin").copy()
        full_index = pd.date_range(
            group["time_bin"].min(), group["time_bin"].max(), freq=TIME_BIN, tz="UTC"
        )
        group = group.set_index("time_bin").reindex(full_index)
        group.index.name = "time_bin"
        for column, value in zip(group_keys, group_values):
            group[column] = value
        completed.append(group.reset_index())

    minute_data = pd.concat(completed, ignore_index=True)
    count_columns = [
        c for c in minute_data.columns
        if c not in group_keys + ["time_bin", "attack_now"]
    ]
    minute_data["has_observation"] = minute_data["event_count"].notna().astype("float32")
    minute_data[count_columns] = minute_data[count_columns].fillna(0)
    minute_data["attack_now"] = minute_data["attack_now"].fillna(0).astype("int8")

    minute_data["hour"] = minute_data["time_bin"].dt.hour.astype("float32")
    minute_data["minute"] = minute_data["time_bin"].dt.minute.astype("float32")
    minute_data["hour_sin"] = np.sin(2 * np.pi * minute_data["hour"] / 24).astype("float32")
    minute_data["hour_cos"] = np.cos(2 * np.pi * minute_data["hour"] / 24).astype("float32")
    minute_data["minute_sin"] = np.sin(2 * np.pi * minute_data["minute"] / 60).astype("float32")
    minute_data["minute_cos"] = np.cos(2 * np.pi * minute_data["minute"] / 60).astype("float32")

    assert not minute_data.duplicated(group_keys + ["time_bin"]).any()
    return minute_data


train_minutes = build_minute_features(train_data)
validation_minutes = build_minute_features(validation_data)
test_minutes = build_minute_features(test_data)

print("Train minute shape:", train_minutes.shape)
print("Validation minute shape:", validation_minutes.shape)
print("Test minute shape:", test_minutes.shape)


Train minute shape: (141019, 62)
Validation minute shape: (74289, 58)
Test minute shape: (22634, 58)


In [48]:
def add_future_target(data):
    output = []
    group_columns = ["dataset_source", "hostname", "split_group", "date"]

    for _, group in data.groupby(group_columns, sort=False, observed=True):
        group = group.sort_values("time_bin").copy()

        minute_differences = group["time_bin"].diff().dropna().dt.total_seconds().div(60)
        assert minute_differences.eq(1).all(), "Non-consecutive minute grid detected."

        future_labels = pd.concat(
            [group["attack_now"].shift(-step) for step in range(1, PREDICT_AHEAD_MINUTES + 1)],
            axis=1,
        )
        # Require the complete future horizon. The last three minutes of a group
        # are unknown rather than incorrectly labelled benign.
        valid_future = future_labels.notna().all(axis=1)
        group["future_attack"] = future_labels.max(axis=1).where(valid_future)
        output.append(group)

    result = pd.concat(output, ignore_index=True)
    result["future_attack"] = result["future_attack"].astype("Float32")
    return result


train_minutes = add_future_target(train_minutes)
validation_minutes = add_future_target(validation_minutes)
test_minutes = add_future_target(test_minutes)

for name, frame in {
    "TRAIN": train_minutes,
    "VALIDATION": validation_minutes,
    "TEST": test_minutes,
}.items():
    known = frame["future_attack"].dropna().astype(int)
    print(f"\n{name} future target")
    print(known.value_counts())
    print("Positive rate:", known.mean())



TRAIN future target
future_attack
0    119525
1     20574
Name: count, dtype: int64
Positive rate: 0.14685329659740612

VALIDATION future target
future_attack
0    64560
1     9358
Name: count, dtype: int64
Positive rate: 0.12659974566411428

TEST future target
future_attack
0    21983
1      469
Name: count, dtype: int64
Positive rate: 0.020889007660787457


### Train Partition Scaling

In [49]:
metadata_columns = [
    "dataset_source", "hostname", "split_group", "date", "time_bin",
    "attack_now", "future_attack",
]
feature_columns = [c for c in train_minutes.columns if c not in metadata_columns]

def align_features(frame):
    aligned = frame.copy()
    for col in feature_columns:
        if col not in aligned.columns:
            aligned[col] = 0
    return (
        aligned[feature_columns]
        .apply(pd.to_numeric, errors="coerce")
        .fillna(0)
        .astype("float32")
    )

X_train_minutes = align_features(train_minutes)
X_validation_minutes = align_features(validation_minutes)
X_test_minutes = align_features(test_minutes)

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train_minutes).astype("float32")
X_validation_scaled = scaler.transform(X_validation_minutes).astype("float32")
X_test_scaled = scaler.transform(X_test_minutes).astype("float32")

print("Number of temporal features:", len(feature_columns))
print("Scaled train shape:", X_train_scaled.shape)
print("Scaled validation shape:", X_validation_scaled.shape)
print("Scaled test shape:", X_test_scaled.shape)


Number of temporal features: 56
Scaled train shape: (141019, 56)
Scaled validation shape: (74289, 56)
Scaled test shape: (22634, 56)


In [50]:
def create_sequences(
    data,
    scaled_features,
    lookback=LOOKBACK_MINUTES
):

    X_sequences = []
    y_sequences = []
    metadata = []

    group_columns = [
        "dataset_source",
        "hostname",
        "split_group",
        "date"
    ]

    working = (
        data
        .reset_index(drop=True)
        .copy()
    )

    assert len(working) == len(scaled_features), (
        "The minute dataframe and scaled feature matrix "
        "have different numbers of rows."
    )

    working["_row_position"] = np.arange(
        len(working)
    )

    for _, group in working.groupby(
        group_columns,
        sort=False,
        observed=True
    ):

        group = (
            group
            .sort_values("time_bin")
            .reset_index(drop=True)
        )

        positions = (
            group["_row_position"]
            .to_numpy()
        )

        labels = (
            group["future_attack"]
            .to_numpy()
        )

        attacks = (
            group["attack_now"]
            .to_numpy()
        )

        times = (
            group["time_bin"]
            .to_numpy()
        )

        source = group[
            "dataset_source"
        ].iloc[0]

        hostname = group[
            "hostname"
        ].iloc[0]

        split_group = group[
            "split_group"
        ].iloc[0]

        group_date = group[
            "date"
        ].iloc[0]

        for end_idx in range(
            lookback - 1,
            len(group)
        ):

            start_idx = (
                end_idx
                - lookback
                + 1
            )

            # Skip rows without a complete
            # future prediction horizon
            if pd.isna(
                labels[end_idx]
            ):
                continue

            # Skip sequences containing an attack
            # in the ten-minute lookback period
            if (
                attacks[
                    start_idx:end_idx + 1
                ].max()
                > 0
            ):
                continue

            actual_span = (
                pd.Timestamp(
                    times[end_idx]
                )
                -
                pd.Timestamp(
                    times[start_idx]
                )
            ).total_seconds() / 60

            # Ensure the ten rows represent
            # ten consecutive clock minutes
            if not np.isclose(
                actual_span,
                lookback - 1
            ):
                continue

            window_positions = positions[
                start_idx:end_idx + 1
            ]

            if (
                len(window_positions)
                != lookback
            ):
                continue

            X_sequences.append(
                scaled_features[
                    window_positions
                ]
            )

            y_sequences.append(
                int(
                    labels[end_idx]
                )
            )

            metadata.append({
                "dataset_source":
                    source,

                "hostname":
                    hostname,

                "split_group":
                    split_group,

                "date":
                    group_date,

                "window_start":
                    times[start_idx],

                "window_end":
                    times[end_idx]
            })


    # Return correctly shaped empty arrays
    # if a partition produces no sequences
    if not X_sequences:

        empty_X = np.empty(
            (
                0,
                lookback,
                scaled_features.shape[1]
            ),
            dtype=np.float32
        )

        empty_y = np.empty(
            0,
            dtype=np.int8
        )

        empty_metadata = pd.DataFrame(
            columns=[
                "dataset_source",
                "hostname",
                "split_group",
                "date",
                "window_start",
                "window_end"
            ]
        )

        return (
            empty_X,
            empty_y,
            empty_metadata
        )


    return (
        np.asarray(
            X_sequences,
            dtype=np.float32
        ),

        np.asarray(
            y_sequences,
            dtype=np.int8
        ),

        pd.DataFrame(
            metadata
        )
    )


# Create training sequences

X_train_seq, Y_train_seq, train_seq_meta = (
    create_sequences(
        train_minutes,
        X_train_scaled
    )
)


# Create validation sequences

X_validation_seq, Y_validation_seq, validation_seq_meta = (
    create_sequences(
        validation_minutes,
        X_validation_scaled
    )
)


# Create test sequences

X_test_seq, Y_test_seq, test_seq_meta = (
    create_sequences(
        test_minutes,
        X_test_scaled
    )
)


# Store partitions for validation

sequence_partitions = [
    (
        "TRAIN",
        X_train_seq,
        Y_train_seq,
        train_seq_meta
    ),

    (
        "VALIDATION",
        X_validation_seq,
        Y_validation_seq,
        validation_seq_meta
    ),

    (
        "TEST",
        X_test_seq,
        Y_test_seq,
        test_seq_meta
    )
]


invalid_partitions = []


# Inspect each partition

for (
    name,
    X_part,
    y_part,
    meta_part
) in sequence_partitions:

    print(
        f"\n===== {name} ====="
    )

    print(
        "Sequence shape:",
        X_part.shape
    )

    assert (
        len(X_part)
        == len(y_part)
        == len(meta_part)
    ), (
        f"{name} feature, label and metadata "
        "lengths do not match."
    )


    if len(y_part) == 0:

        print(
            "No valid sequences were created."
        )

        invalid_partitions.append(
            name
        )

        continue


    target_series = pd.Series(
        y_part,
        index=meta_part.index,
        name="target"
    )


    print(
        "\nTarget distribution:"
    )

    print(
        target_series
        .value_counts()
        .sort_index()
    )


    print(
        "\nTarget distribution by source:"
    )

    print(
        pd.crosstab(
            meta_part[
                "dataset_source"
            ],

            target_series
        )
    )


    print(
        "\nSequences by split group:"
    )

    group_diagnostics = (
        meta_part
        .assign(
            target=target_series
        )
        .groupby(
            [
                "dataset_source",
                "split_group"
            ],
            observed=True
        )
        .agg(
            sequences=(
                "target",
                "size"
            ),

            positives=(
                "target",
                "sum"
            ),

            unique_labels=(
                "target",
                "nunique"
            )
        )
        .sort_values(
            [
                "positives",
                "sequences"
            ],
            ascending=False
        )
    )

    print(
        group_diagnostics.head(20)
    )


    spans = (
        pd.to_datetime(
            meta_part[
                "window_end"
            ]
        )
        -
        pd.to_datetime(
            meta_part[
                "window_start"
            ]
        )
    ).dt.total_seconds().div(60)


    assert np.isclose(
        spans,
        LOOKBACK_MINUTES - 1
    ).all(), (
        f"{name} contains sequences that do not "
        f"cover exactly {LOOKBACK_MINUTES} minutes."
    )


    if (
        target_series.nunique()
        < 2
    ):

        invalid_partitions.append(
            name
        )

        available_class = (
            int(target_series.iloc[0])
        )

        print(
            f"\nWARNING: {name} contains "
            f"only class {available_class}."
        )


# Do not start model training when any
# partition contains only one class

if invalid_partitions:

    raise ValueError(
        "The following sequence partitions do not "
        "contain both classes: "
        + ", ".join(invalid_partitions)
        + ". Recreate the grouped split using groups "
        "that produce valid positive and negative "
        "temporal sequences."
    )


print(
    "\nAll partitions contain both classes, "
    "and all temporal checks passed."
)


===== TRAIN =====
Sequence shape: (103970, 10, 56)

Target distribution:
target
0    100574
1      3396
Name: count, dtype: int64

Target distribution by source:
target              0     1
dataset_source             
OpTC             2448    15
Windows-APT     98126  3381

Sequences by split group:
                                                                sequences  \
dataset_source split_group                                                  
Windows-APT    Windows-APT|scenario|S35                             21232   
               Windows-APT|scenario|S07                             17011   
               Windows-APT|scenario|S33                             14153   
               Windows-APT|scenario|S28                              9370   
               Windows-APT|scenario|S16                              3346   
               Windows-APT|scenario|S32                              3097   
               Windows-APT|scenario|S25                              1116   
     

### Test Partition Scaling

In [51]:
classes = np.unique(Y_train_seq)
weights = compute_class_weight(class_weight="balanced", classes=classes, y=Y_train_seq)
balanced_weights = {int(cls): float(weight) for cls, weight in zip(classes, weights)}
class_weights = {cls: float(np.sqrt(weight)) for cls, weight in balanced_weights.items()}

print("Balanced weights:", balanced_weights)
print("Used weights:", class_weights)

X_train_temporal = X_train_seq.reshape(X_train_seq.shape[0], -1)
X_validation_temporal = X_validation_seq.reshape(X_validation_seq.shape[0], -1)
X_test_temporal = X_test_seq.reshape(X_test_seq.shape[0], -1)

Balanced weights: {0: 0.5168830910573309, 1: 15.30771495877503}
Used weights: {0: 0.7189458192780113, 1: 3.91250750271166}


### Classical ML models

In [52]:
# Classical models are fitted on training data only.
lgr_model = LogisticRegression(
    max_iter=1000, random_state=seed, class_weight="balanced", n_jobs=-1
)
lgr_model.fit(X_train_temporal, Y_train_seq)

xgb_model = XGBClassifier(
    random_state=seed,
    n_estimators=300,
    max_depth=4,
    learning_rate=0.03,
    subsample=0.8,
    colsample_bytree=0.8,
    min_child_weight=3,
    reg_alpha=0.1,
    reg_lambda=1.0,
    eval_metric="logloss",
    n_jobs=-1,
)
xgb_model.fit(X_train_temporal, Y_train_seq)

XGBClassifier(base_score=None, booster=None, callbacks=None,
              colsample_bylevel=None, colsample_bynode=None,
              colsample_bytree=0.8, device=None, early_stopping_rounds=None,
              enable_categorical=True, eval_metric='logloss',
              feature_types=None, feature_weights=None, gamma=None,
              grow_policy=None, importance_type=None,
              interaction_constraints=None, learning_rate=0.03, max_bin=None,
              max_cat_threshold=None, max_cat_to_onehot=None,
              max_delta_step=None, max_depth=4, max_leaves=None,
              min_child_weight=3, missing=nan, monotone_constraints=None,
              multi_strategy=None, n_estimators=300, n_jobs=-1,
              num_parallel_tree=None, ...)

### Temporal Models

In [53]:
# LSTM

lstm_model = Sequential([
    Input(
        shape=(
            X_train_seq.shape[1],
            X_train_seq.shape[2]
        )
    ),

    LSTM(
        128,
        return_sequences=True
    ),

    BatchNormalization(),
    Dropout(0.25),

    LSTM(64),

    BatchNormalization(),
    Dropout(0.25),

    Dense(
        64,
        activation="relu"
    ),

    Dropout(0.20),

    Dense(
        32,
        activation="relu"
    ),

    Dense(
        1,
        activation="sigmoid"
    )
])

lstm_model.compile(
    optimizer=Adam(
        learning_rate=0.0005
    ),
    loss="binary_crossentropy",
    metrics=[
        "accuracy",
        tf.keras.metrics.Precision(name="precision"),
        tf.keras.metrics.Recall(name="recall"),
        tf.keras.metrics.AUC(name="auc"),
        tf.keras.metrics.AUC(name="pr_auc", curve="PR")
    ]
)

lstm_callbacks = [
    EarlyStopping(
        monitor="val_loss",
        patience=5,
        restore_best_weights=True
    ),
    ReduceLROnPlateau(
        monitor="val_loss",
        factor=0.5,
        patience=3,
        min_lr=1e-6
    )
]

lstm_history = lstm_model.fit(
    X_train_seq,
    Y_train_seq,
    validation_data=(
        X_validation_seq,
        Y_validation_seq
    ),
    epochs=30,
    batch_size=64,
    class_weight=class_weights,
    callbacks=lstm_callbacks,
    shuffle=False,
    verbose=1
)


Epoch 1/30
1625/1625 ━━━━━━━━━━━━━━━━━━━━ 97s 57ms/step - accuracy: 0.9570 - auc: 0.5688 - loss: 0.3750 - pr_auc: 0.0483 - precision: 0.1007 - recall: 0.0400 - val_accuracy: 0.7735 - val_auc: 0.5184 - val_loss: 0.4227 - val_pr_auc: 0.0221 - val_precision: 0.0223 - val_recall: 0.2234 - learning_rate: 5.0000e-04
Epoch 2/30
1625/1625 ━━━━━━━━━━━━━━━━━━━━ 87s 54ms/step - accuracy: 0.9565 - auc: 0.6437 - loss: 0.3467 - pr_auc: 0.0751 - precision: 0.1440 - recall: 0.0671 - val_accuracy: 0.6864 - val_auc: 0.5294 - val_loss: 0.5196 - val_pr_auc: 0.0246 - val_precision: 0.0258 - val_recall: 0.3707 - learning_rate: 5.0000e-04
Epoch 3/30
1625/1625 ━━━━━━━━━━━━━━━━━━━━ 89s 55ms/step - accuracy: 0.9539 - auc: 0.7110 - loss: 0.3257 - pr_auc: 0.0970 - precision: 0.1679 - recall: 0.1042 - val_accuracy: 0.5780 - val_auc: 0.5442 - val_loss: 0.5972 - val_pr_auc: 0.0293 - val_precision: 0.0222 - val_recall: 0.4342 - learning_rate: 5.0000e-04
Epoch 4/30
1625/1625 ━━━━━━━━━━━━━━━━━━━━ 87s 54ms/step - accura

In [54]:
# GRU

gru_model = Sequential([
    Input(
        shape=(
            X_train_seq.shape[1],
            X_train_seq.shape[2]
        )
    ),

    GRU(
        128,
        return_sequences=True
    ),

    BatchNormalization(),
    Dropout(0.25),

    GRU(64),

    BatchNormalization(),
    Dropout(0.25),

    Dense(
        64,
        activation="relu"
    ),

    Dropout(0.20),

    Dense(
        32,
        activation="relu"
    ),

    Dense(
        1,
        activation="sigmoid"
    )
])

gru_model.compile(
    optimizer=Adam(
        learning_rate=0.0005
    ),
    loss="binary_crossentropy",
    metrics=[
        "accuracy",
        tf.keras.metrics.Precision(name="precision"),
        tf.keras.metrics.Recall(name="recall"),
        tf.keras.metrics.AUC(name="auc"),
        tf.keras.metrics.AUC(name="pr_auc", curve="PR")
    ]
)

gru_callbacks = [
    EarlyStopping(
        monitor="val_loss",
        patience=5,
        restore_best_weights=True
    ),
    ReduceLROnPlateau(
        monitor="val_loss",
        factor=0.5,
        patience=3,
        min_lr=1e-6
    )
]

gru_history = gru_model.fit(
    X_train_seq,
    Y_train_seq,
    validation_data=(
        X_validation_seq,
        Y_validation_seq
    ),
    epochs=30,
    batch_size=64,
    class_weight=class_weights,
    callbacks=gru_callbacks,
    shuffle=False,
    verbose=1
)


Epoch 1/30
1625/1625 ━━━━━━━━━━━━━━━━━━━━ 96s 55ms/step - accuracy: 0.9529 - auc: 0.5796 - loss: 0.3728 - pr_auc: 0.0498 - precision: 0.0954 - recall: 0.0521 - val_accuracy: 0.8346 - val_auc: 0.5183 - val_loss: 0.4544 - val_pr_auc: 0.0218 - val_precision: 0.0154 - val_recall: 0.1066 - learning_rate: 5.0000e-04
Epoch 2/30
1625/1625 ━━━━━━━━━━━━━━━━━━━━ 98s 60ms/step - accuracy: 0.9565 - auc: 0.6762 - loss: 0.3374 - pr_auc: 0.0836 - precision: 0.1575 - recall: 0.0766 - val_accuracy: 0.6842 - val_auc: 0.5214 - val_loss: 0.5688 - val_pr_auc: 0.0217 - val_precision: 0.0225 - val_recall: 0.3245 - learning_rate: 5.0000e-04
Epoch 3/30
1625/1625 ━━━━━━━━━━━━━━━━━━━━ 90s 56ms/step - accuracy: 0.9519 - auc: 0.7336 - loss: 0.3168 - pr_auc: 0.1065 - precision: 0.1669 - recall: 0.1181 - val_accuracy: 0.4802 - val_auc: 0.5020 - val_loss: 0.7244 - val_pr_auc: 0.0203 - val_precision: 0.0225 - val_recall: 0.5470 - learning_rate: 5.0000e-04
Epoch 4/30
1625/1625 ━━━━━━━━━━━━━━━━━━━━ 86s 53ms/step - accura

In [55]:
# Thresholds are selected exclusively on validation macro F1.
def choose_threshold(y_true, probabilities):
    candidates = np.round(np.arange(0.10, 0.901, 0.01), 2)
    records = []
    for threshold in candidates:
        predictions = (probabilities >= threshold).astype(int)
        records.append({
            "threshold": threshold,
            "macro_f1": metrics.f1_score(y_true, predictions, average="macro", zero_division=0),
            "attack_f1": metrics.f1_score(y_true, predictions, pos_label=1, zero_division=0),
            "attack_recall": metrics.recall_score(y_true, predictions, pos_label=1, zero_division=0),
        })
    table = pd.DataFrame(records)
    best_score = table["macro_f1"].max()
    # If tied, choose the threshold closest to 0.5.
    best_rows = table.loc[np.isclose(table["macro_f1"], best_score)].copy()
    best_rows["distance_from_half"] = (best_rows["threshold"] - 0.5).abs()
    best = best_rows.sort_values(["distance_from_half", "threshold"]).iloc[0]
    return float(best["threshold"]), table


validation_probabilities = {
    "Logistic Regression": lgr_model.predict_proba(X_validation_temporal)[:, 1],
    "XGBoost": xgb_model.predict_proba(X_validation_temporal)[:, 1],
    "LSTM": lstm_model.predict(X_validation_seq, verbose=0).ravel(),
    "GRU": gru_model.predict(X_validation_seq, verbose=0).ravel(),
}

thresholds = {}
threshold_tables = {}
for model_name, probabilities in validation_probabilities.items():
    thresholds[model_name], threshold_tables[model_name] = choose_threshold(
        Y_validation_seq, probabilities
    )

LGR_THRESHOLD = thresholds["Logistic Regression"]
XGB_THRESHOLD = thresholds["XGBoost"]
LSTM_THRESHOLD = thresholds["LSTM"]
THRESHOLD = thresholds["GRU"]  # Preserve the previous GRU variable name.

print("Validation-selected thresholds:")
print(pd.Series(thresholds).sort_index())


Validation-selected thresholds:
GRU                    0.57
LSTM                   0.62
Logistic Regression    0.64
XGBoost                0.10
dtype: float64


In [56]:
# Evaluate Partition
def probability_metrics(y_true, probabilities, threshold):
    predictions = (probabilities >= threshold).astype(int)
    result = {
        "Threshold": threshold,
        "Accuracy": metrics.accuracy_score(y_true, predictions),
        "Balanced Accuracy": metrics.balanced_accuracy_score(y_true, predictions),
        "Attack Precision": metrics.precision_score(y_true, predictions, pos_label=1, zero_division=0),
        "Attack Recall": metrics.recall_score(y_true, predictions, pos_label=1, zero_division=0),
        "Attack F1": metrics.f1_score(y_true, predictions, pos_label=1, zero_division=0),
        "Macro F1": metrics.f1_score(y_true, predictions, average="macro", zero_division=0),
        "ROC AUC": metrics.roc_auc_score(y_true, probabilities),
        "PR AUC": metrics.average_precision_score(y_true, probabilities),
    }
    return result, predictions


def evaluate_partition(name, X_seq, X_temporal, y_true):
    probabilities = {
        "Logistic Regression": lgr_model.predict_proba(X_temporal)[:, 1],
        "XGBoost": xgb_model.predict_proba(X_temporal)[:, 1],
        "LSTM": lstm_model.predict(X_seq, verbose=0).ravel(),
        "GRU": gru_model.predict(X_seq, verbose=0).ravel(),
    }

    rows, predictions = [], {}
    for model_name, model_probabilities in probabilities.items():
        row, predictions[model_name] = probability_metrics(
            y_true, model_probabilities, thresholds[model_name]
        )
        row["Model"] = model_name
        rows.append(row)

    table = pd.DataFrame(rows).set_index("Model")
    baseline_accuracy = max(np.mean(y_true == 0), np.mean(y_true == 1))
    return table, probabilities, predictions, baseline_accuracy


def show_partition_results(name, table, baseline_accuracy):
    print(f"\n{name} majority-class baseline accuracy: {baseline_accuracy:.4f}")
    display(table.round(4))

In [57]:
train_results, train_probabilities, train_predictions, train_baseline = evaluate_partition(
    "TRAIN", X_train_seq, X_train_temporal, Y_train_seq
)

validation_results, validation_probabilities, validation_predictions, validation_baseline = evaluate_partition(
    "VALIDATION", X_validation_seq, X_validation_temporal, Y_validation_seq
)

# Final test evaluation: thresholds and models are already frozen.
test_results, test_probabilities, test_predictions, test_baseline = evaluate_partition(
    "TEST", X_test_seq, X_test_temporal, Y_test_seq
)


### Result Presentation

In [58]:
# Confusion matrices and reports for the final test set
for model_name, predictions in test_predictions.items():
    print(f"\n===== {model_name} - FINAL TEST =====")
    print("Confusion matrix:\n", metrics.confusion_matrix(Y_test_seq, predictions))
    print(metrics.classification_report(
        Y_test_seq, predictions, digits=4, zero_division=0
    ))



===== Logistic Regression - FINAL TEST =====
Confusion matrix:
 [[19620  1534]
 [  112    10]]
              precision    recall  f1-score   support

           0     0.9943    0.9275    0.9597     21154
           1     0.0065    0.0820    0.0120       122

    accuracy                         0.9226     21276
   macro avg     0.5004    0.5047    0.4859     21276
weighted avg     0.9887    0.9226    0.9543     21276


===== XGBoost - FINAL TEST =====
Confusion matrix:
 [[20829   325]
 [  121     1]]
              precision    recall  f1-score   support

           0     0.9942    0.9846    0.9894     21154
           1     0.0031    0.0082    0.0045       122

    accuracy                         0.9790     21276
   macro avg     0.4986    0.4964    0.4969     21276
weighted avg     0.9885    0.9790    0.9838     21276


===== LSTM - FINAL TEST =====
Confusion matrix:
 [[20488   666]
 [  114     8]]
              precision    recall  f1-score   support

           0     0.9945    0.9

In [59]:
show_partition_results("TRAIN", train_results, train_baseline)
show_partition_results("VALIDATION", validation_results, validation_baseline)
show_partition_results("TEST", test_results, test_baseline)


TRAIN majority-class baseline accuracy: 0.9673


,Threshold,Accuracy,Balanced Accuracy,Attack Precision,Attack Recall,Attack F1,Macro F1,ROC AUC,PR AUC
Model,,,,,,,,,
Logistic Regression,0.64,0.9234,0.5409,0.0819,0.1316,0.1010,0.5305,0.6187,0.0631
XGBoost,0.10,0.9586,0.5413,0.2071,0.0948,0.1301,0.5544,0.7050,0.1123
LSTM,0.62,0.9371,0.5019,0.0363,0.0362,0.0363,0.5019,0.5277,0.0350
GRU,0.57,0.9579,0.4977,0.0177,0.0053,0.0082,0.4933,0.5185,0.0330



VALIDATION majority-class baseline accuracy: 0.9786


,Threshold,Accuracy,Balanced Accuracy,Attack Precision,Attack Recall,Attack F1,Macro F1,ROC AUC,PR AUC
Model,,,,,,,,,
Logistic Regression,0.64,0.9346,0.5615,0.0717,0.1716,0.1011,0.5336,0.6443,0.0419
XGBoost,0.10,0.9647,0.5178,0.0678,0.0509,0.0582,0.5201,0.6593,0.0440
LSTM,0.62,0.9458,0.5009,0.0225,0.0361,0.0277,0.4999,0.5183,0.0222
GRU,0.57,0.9694,0.4984,0.0142,0.0063,0.0087,0.4966,0.5179,0.0221



TEST majority-class baseline accuracy: 0.9943


,Threshold,Accuracy,Balanced Accuracy,Attack Precision,Attack Recall,Attack F1,Macro F1,ROC AUC,PR AUC
Model,,,,,,,,,
Logistic Regression,0.64,0.9226,0.5047,0.0065,0.0820,0.0120,0.4859,0.5031,0.0089
XGBoost,0.10,0.9790,0.4964,0.0031,0.0082,0.0045,0.4969,0.5615,0.0073
LSTM,0.62,0.9633,0.5170,0.0119,0.0656,0.0201,0.5007,0.5459,0.0075
GRU,0.57,0.9849,0.4953,0.0000,0.0000,0.0000,0.4962,0.5960,0.0071


In [60]:
# Sequence distribution report
audit_rows = []
for name, y_part, meta_part in [
    ("Train", Y_train_seq, train_seq_meta),
    ("Validation", Y_validation_seq, validation_seq_meta),
    ("Test", Y_test_seq, test_seq_meta),
]:
    spans = (
        pd.to_datetime(meta_part["window_end"]) - pd.to_datetime(meta_part["window_start"])
    ).dt.total_seconds().div(60)
    audit_rows.append({
        "Split": name,
        "Sequences": len(y_part),
        "Positive sequences": int(y_part.sum()),
        "Positive rate": float(y_part.mean()),
        "Majority baseline accuracy": float(max(np.mean(y_part == 0), np.mean(y_part == 1))),
        "Independent groups": int(meta_part["split_group"].nunique()),
        "Minimum window span": float(spans.min()),
        "Maximum window span": float(spans.max()),
    })

audit_table = pd.DataFrame(audit_rows)
display(audit_table.round(4))

assert set(train_seq_meta["split_group"]).isdisjoint(validation_seq_meta["split_group"])
assert set(train_seq_meta["split_group"]).isdisjoint(test_seq_meta["split_group"])
assert set(validation_seq_meta["split_group"]).isdisjoint(test_seq_meta["split_group"])
assert audit_table["Minimum window span"].eq(LOOKBACK_MINUTES - 1).all()
assert audit_table["Maximum window span"].eq(LOOKBACK_MINUTES - 1).all()

print("All final group-independence and temporal-continuity checks passed.")

,Split,Sequences,Positive sequences,Positive rate,Majority baseline accuracy,Independent groups,Minimum window span,Maximum window span
0,Train,103970,3396,0.0327,0.9673,59,9.0,9.0
1,Validation,59559,1276,0.0214,0.9786,22,9.0,9.0
2,Test,21276,122,0.0057,0.9943,19,9.0,9.0


All final group-independence and temporal-continuity checks passed.
